# Extracción de cursos por carrera

In [1]:
from selenium import webdriver
from time import sleep
import random
from bs4 import BeautifulSoup as bs
from selenium.common.exceptions import NoSuchElementException


In [2]:
url = 'https://www.up.edu.pe/'

In [3]:
htmls = list()

In [4]:
driver = webdriver.Chrome()
driver.maximize_window()
driver.set_page_load_timeout(60)
driver.implicitly_wait(10) 

driver.get(url)
driver.find_element(by = 'xpath', value = '//*[@id="ctl00_ctl40_g_b7ddf07a_5919_4611_be5a_923d1f4ceb0c"]/div/div/div[1]/div/a/img').click()

import time
from selenium.webdriver.common.by import By

for i in range(1, 13):
    # 1. Entrar a la carrera i
    xpath = f'//*[@id="ctl00_ctl40_g_41ed8ed3_3073_4069_9c40_345d6c91bdc4_WPCajas"]/a[{i}]/div[1]/img'
    driver.find_element(By.XPATH, xpath).click()
    time.sleep(2)  # esperar a que cargue la página

    # 2. --- AQUÍ HACES TUS COSAS DENTRO DE LA CARRERA ---
    print(f"Carrera {i}: {driver.title}")

    driver.find_element(By.LINK_TEXT, "Cursos de tu carrera").click()

    time.sleep(2)  # esperar a que cargue la página
    html = driver.page_source

    htmls.append(html)

    # 3. Volver a la lista de carreras
    driver.back()
    time.sleep(2)
    driver.back()
    time.sleep(1)

Carrera 1: Ingeniería en Innovación y Diseño | Pregrado | Universidad del Pacífico
Carrera 2: Humanidades Digitales | Pregrado | Universidad del Pacífico
Carrera 3: Política, Filosofía y Economía | Pregrado | Universidad del Pacífico
Carrera 4: Administración | Pregrado | Universidad del Pacífico
Carrera 5: Contabilidad | Pregrado | Universidad del Pacífico
Carrera 6: Derecho | Pregrado | Universidad del Pacífico
Carrera 7: Economía | Pregrado | Universidad del Pacífico
Carrera 8: Finanzas | Pregrado | Universidad del Pacífico
Carrera 9: Ingeniería Empresarial | Pregrado | Universidad del Pacífico
Carrera 10: Ingeniería de la Información | Pregrado | UP
Carrera 11: Marketing | Pregrado | Universidad del Pacífico
Carrera 12: Negocios Internacionales | Pregrado | UP


In [5]:
import pandas as pd
from bs4 import BeautifulSoup

datos = []

for i, html in enumerate(htmls, start=1):
    soup = BeautifulSoup(html, "html.parser")
    carrera = soup.title.get_text(strip=True) if soup.title else f"Carrera {i}"

    for header in soup.select("div.collapsible-header"):
        if header.get_text(strip=True).startswith("Ciclo"):
            body = header.find_next_sibling("div", class_="collapsible-body")
            if body:
                for li in body.select("ul li"):
                    curso = li.get_text(" ", strip=True).replace("\u200b", "").replace("\xa0", " ").strip()
                    if curso:
                        datos.append({"carrera": carrera, "curso": curso})

df = pd.DataFrame(datos)
df

,carrera,curso
0,Plan de Estudios | Carrera de Ingeniería en In...,Nivelación en Lenguaje
1,Plan de Estudios | Carrera de Ingeniería en In...,Nivelación en Matemáticas
2,Plan de Estudios | Carrera de Ingeniería en In...,Nivelación en Informática
3,Plan de Estudios | Carrera de Ingeniería en In...,Introducción a la Ingeniería
4,Plan de Estudios | Carrera de Ingeniería en In...,Matemáticas I
...,...,...
589,Plan de Estudios | Negocios Internacionales | UP,Estrategias de Internacionalización
590,Plan de Estudios | Negocios Internacionales | UP,International Human Resources Management
591,Plan de Estudios | Negocios Internacionales | UP,Proyección Social
592,Plan de Estudios | Negocios Internacionales | UP,International Business Plan


# Extracción de sumillas

In [6]:
import subprocess
import pandas as pd

In [7]:
pdf = "Sumillas-de-asignaturas-obligatorias_Plan-de-Estudios-2022.pdf"

# En el índice, el Departamento Académico de Ingeniería va de la pág. 83 a la 102
texto = subprocess.check_output(["pdftotext", "-f", "83", "-l", "102", pdf, "-"], text=True)
lineas = [
    l.strip() for l in texto.splitlines()
    if l.strip() and l.strip() != "UNIVERSIDAD DEL PACÍFICO" and not l.strip().isdigit()
]

filas = []
curso_actual = None
desc = []
esperando_nombre = False

for l in lineas:
    if l == "CURSO":
        if curso_actual:
            filas.append({
                "universidad": "UP",
                "departamento": "Ingeniería",
                "curso": curso_actual,
                "descripcion": " ".join(desc),
            })
        esperando_nombre = True
        desc = []
    elif esperando_nombre:
        curso_actual = l
        esperando_nombre = False
    elif curso_actual:
        desc.append(l)

if curso_actual:
    filas.append({
        "universidad": "UP",
        "departamento": "Ingeniería",
        "curso": curso_actual,
        "descripcion": " ".join(desc),
    })

df_up = pd.DataFrame(filas)
print(f"Total cursos extraídos: {len(df_up)}")


Total cursos extraídos: 41


# Emparejamiento de habilidades

In [8]:
import re

HABILIDADES = {
    # Programación y Datos
    "POO": ["poo", "programación orientada a objetos", "orientada a objetos"],
    "SQL / Bases de Datos": ["sql", "base de datos", "bases de datos", "mysql", "postgresql", "oracle", "modelamiento de datos"],
    "NoSQL": ["nosql", "mongodb"],
    "Python": ["python"],
    "Java": ["java"],
    "C/C++": ["c++", "lenguaje c"],
    "JavaScript / Web": ["javascript", "html", "css", "desarrollo web", "aplicaciones web"],
    "Desarrollo Móvil": ["móvil", "móviles", "android"],
    "Algoritmos y Estructuras": ["algoritmos", "estructuras de datos"],
    "Git": ["git", "control de versiones"],
    "Linux": ["linux", "unix"],
    "Sistemas Operativos": ["sistemas operativos", "sistema operativo"],
    "Cloud Computing": ["cloud", "nube", "aws", "azure"],
    "Big Data / Minería de Datos": ["big data", "minería de datos", "data mining", "ciencia de datos", "analítica de datos"],
    "Machine Learning / IA": ["machine learning", "aprendizaje automático", "inteligencia artificial", "redes neuronales"],
    "Ciberseguridad": ["ciberseguridad", "seguridad informática", "seguridad de la información", "criptografía"],
    "Redes": ["redes", "tcp/ip", "cisco", "enrutamiento", "conmutación"],
    # Electrónica, Control y Mecatrónica
    "MATLAB / Simulink": ["matlab", "simulink"],
    "AutoCAD / CAD": ["autocad", "cad", "solidworks", "diseño asistido"],
    "PLC / Automatización": ["plc", "autómatas programables", "automatización", "scada"],
    "Arduino": ["arduino"],
    "Sistemas Embebidos": ["embebidos", "microcontroladores", "microprocesadores"],
    "FPGA / VHDL": ["fpga", "vhdl"],
    "Control Automático": ["control automático", "sistemas de control", "pid"],
    "Sensores e Instrumentación": ["sensores", "actuadores", "instrumentación"],
    "Neumática e Hidráulica": ["neumática", "hidráulica", "electroneumática"],
    "Electrónica": ["electrónica", "circuitos"],
    "Telecomunicaciones": ["telecomunicaciones", "señales", "antenas", "fibra óptica"],
    "IoT": ["iot", "internet de las cosas"],
    "Robótica": ["robótica", "robots"],
    # Industrial, Gestión y Analítica
    "Excel": ["excel"],
    "Power BI": ["power bi", "inteligencia de negocios", "business intelligence"],
    "ERP / SAP": ["erp", "sap"],
    "Estadística": ["estadística", "probabilidades"],
    "Investigación de Operaciones": ["investigación de operaciones", "programación lineal", "optimización"],
    "Lean / Six Sigma": ["lean", "six sigma", "mejora continua"],
    "Gestión de Calidad": ["gestión de calidad", "calidad total", "iso 9001"],
    "Logística / Supply Chain": ["logística", "cadena de suministro", "supply chain", "inventarios"],
    "Gestión de Proyectos": ["gestión de proyectos", "pmbok", "scrum", "ágil", "ágiles"],
    "Seguridad Industrial": ["seguridad industrial", "salud ocupacional", "sst"],
    "Costos y Finanzas": ["costos", "presupuestos", "evaluación de proyectos", "finanzas"],
    # Transversales e Idiomas
    "Inglés": ["inglés"],
    "Trabajo en Equipo": ["trabajo en equipo"],
    "Liderazgo": ["liderazgo"],
    "Comunicación Efectiva": ["comunicación efectiva", "comunicación oral y escrita"],
}


def buscar_habilidades(texto):
    texto = str(texto).lower()
    encontradas = []
    for hab, variantes in HABILIDADES.items():
        for v in variantes:
            if re.search(rf"\b{re.escape(v)}\b", texto):
                encontradas.append(hab)
                break
    return encontradas


df_up["habilidades"] = df_up["descripcion"].apply(buscar_habilidades)
df_up[df_up["habilidades"].str.len() > 0][["curso", "habilidades"]].head(10)

,curso,habilidades
0,Álgebra Lineal Aplicada,"[Algoritmos y Estructuras, Estadística, Invest..."
1,Analítica de la Web,"[Big Data / Minería de Datos, Machine Learning..."
2,Arquitectura del Sistema de Información,"[Power BI, Liderazgo, Comunicación Efectiva]"
3,Big Data Analytics,"[Big Data / Minería de Datos, Machine Learning..."
4,Business Intelligence,"[Cloud Computing, Power BI, Liderazgo]"
5,Computación de Alto Desempeño y Cloud Computing,"[Cloud Computing, Big Data / Minería de Datos,..."
6,Data Mining,"[Python, Big Data / Minería de Datos, Liderazgo]"
7,Deep Learning,"[Algoritmos y Estructuras, Machine Learning / ..."
8,Desarrollo de Soluciones Empresariales,[Liderazgo]
9,Design Thinking and Technological Innovation,"[Liderazgo, Comunicación Efectiva]"


In [9]:
df.to_csv("carreras_cursos_up.csv", index=False, encoding="utf-8")